In [ ]:
import json
import urllib.error
import urllib.request
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import scipy.stats as stats
import itertools

import altair as alt
from sklearn.metrics import precision_recall_curve, auc
from natsort import natsorted

import seaborn as sns

alt.data_transformers.disable_max_rows()

# Overhead for finding data

In [ ]:
def spliceai_mapper(df):

    df['maxSpliceAI'] = df[['spliceAI_DS_AG', 'spliceAI_DS_AL', 'spliceAI_DS_DG', 'spliceAI_DS_DL']].max(axis = 1)

    df['splice_impact'] = '< Threshold'
    SPLICE_IMPACT_COLS = {
        'spliceAI_DS_AG': 'Acceptor Gain',
        'spliceAI_DS_AL': 'Acceptor Loss',
        'spliceAI_DS_DG': 'Donor Gain',
        'spliceAI_DS_DL': 'Donor Loss',
    }

    for col, label in SPLICE_IMPACT_COLS.items():
        df.loc[df[col] >= 0.2, 'splice_impact'] = label
    
    return df

In [ ]:
input_directory = Path('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/sge_data_for_qc')
scores_excel = Path('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/20260101_SGEsubset.xlsx')
rna_dir = Path('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/sge_data_for_qc/raw_scores_for_rna')

all_scores = pd.read_excel(scores_excel)
all_scores['maxSpliceAI'] = all_scores[['spliceAI_DS_AG', 'spliceAI_DS_AL', 'spliceAI_DS_DG', 'spliceAI_DS_DL']].max(axis = 1)

all_scores = spliceai_mapper(all_scores)


In [ ]:
def molecular_consequence_mapper(df, remap_col):

    df = df.dropna(subset=[remap_col]).copy()
    CONSEQUENCE_EXACT = {
            'synonymous_variant': 'Synonymous',
            'intron_variant':     'Intron',
            'stop_gained':        'Stop Gained',
            'stop_lost':          'Stop Lost',
            'start_lost':         'Start Lost',
            'inframe_indel':      'Inframe Indel',
        }

    CONSEQUENCE_CONTAINS = {
        'missense': 'Missense',
        'site':     'Canonical Splice',
        'ing_var':  'Splice Region',
        'UTR':      'UTR Variant',
    }

    df[remap_col] = df[remap_col].replace(CONSEQUENCE_EXACT)
    for pattern, label in CONSEQUENCE_CONTAINS.items():
        df.loc[df[remap_col].str.contains(pattern), remap_col] = label
    

    return df

In [ ]:
def find_genes(input_dir: Path | None = None) -> dict:
    """Discover all gene datasets in the input directory.

    Detects genes by finding all *delcounts.tsv files and extracting the gene
    name from each filename. Both dot-separated (e.g. CTCF.delcounts.tsv) and
    run-together (e.g. 20260129_RAD51Ddelcounts.tsv) naming conventions are
    supported. Companion files (*snvcounts.tsv, *editrates.tsv) must also be
    present for each detected gene. Per-gene scores are sourced from a shared
    Excel file (filtered by the 'Gene' column) passed via scores_excel.

    Args:
        input_dir: Directory containing gene-specific TSV files.
        scores_excel: Path to the shared scores Excel file (e.g.
            20260101_SGEsubset.xlsx). Scores for each gene are obtained by
            filtering rows where Gene == gene_name.

    Returns a dict mapping gene name -> files dict, e.g.:
        {"RAD51D": {"del_counts": Path(...), "snv_counts": Path(...), ...}}
    """
    delcounts_files = sorted(input_dir.glob("*delcounts.tsv"))
    if not delcounts_files:
        raise FileNotFoundError(f"No '*delcounts.tsv' files found in {input_dir}")

    def find_one(*patterns):
        for pattern in patterns:
            matches = list(input_dir.glob(pattern))
            if len(matches) == 1:
                return matches[0]
            if len(matches) > 1:
                raise ValueError(
                    f"Multiple files match '{pattern}': "
                    + ", ".join(str(m) for m in matches)
                )
        raise FileNotFoundError(
            f"Could not find any of {patterns} in {input_dir}"
        )

    genes = {}
    for delcounts_path in delcounts_files:
        # Handles both GENE.delcounts.tsv and GENEdelcounts.tsv (with optional prefix)
        stem_part = delcounts_path.stem.split("_")[-1]
        gene = stem_part.removesuffix(".delcounts").removesuffix("delcounts")

        gene_score_df = all_scores.loc[all_scores['Gene'] == gene].copy()
        gene_score_df = molecular_consequence_mapper(gene_score_df, 'simplified_consequence')

        simple_clinvar_mapping = {
                'Pathogenic': 'Pathogenic',
                'Benign': 'Benign',
                'Likely benign': 'Benign',
                'Benign/Likely benign': 'Benign',
                'Likely pathogenic': 'Pathogenic',
                'Pathogenic/Likely pathogenic': 'Pathogenic',
                'Uncertain significance': "VUS",
                'not provided': 'VUS',
                'Conflicting classifications of pathogenicity': 'VUS',
                'no classification for the single variant': "VUS"
            }
        
        full_clinvar_mapping = {
                'Pathogenic': 'Pathogenic',
                'Benign': 'Benign',
                'Likely benign': 'Likely Benign',
                'Benign/Likely benign': 'Likely Benign',
                'Likely pathogenic': 'Likely Pathogenic',
                'Pathogenic/Likely pathogenic': 'Likely Pathogenic',
                'Uncertain significance': "VUS",
                'not provided': 'VUS',
                'Conflicting classifications of pathogenicity': 'VUS',
                'no classification for the single variant': "VUS"
            }
        
        gene_score_df['simple_clinvar_sig_2025'] = gene_score_df['clinvar_sig_2025'].map(simple_clinvar_mapping)
        gene_score_df['clinvar_sig_2025'] = gene_score_df['clinvar_sig_2025'].map(full_clinvar_mapping)

        genes[gene] = {
            "del_counts": delcounts_path,
            "snv_counts": find_one(f"*{gene}snvcounts.tsv", f"*{gene}.snvcounts.tsv"),
            "edit_rates": find_one(f"*{gene}editrates.tsv", f"*{gene}.editrates.tsv"),
            "scores_excel": gene_score_df
        }

    return genes

In [ ]:
def get_rna(rna_dir, genes):
    rna_dfs = {}
    matches=list(rna_dir.glob("*allscores*"))

    for match in matches:
        gene = str(match).split('.')[0].split('/')[-1]

        if gene in genes.keys():
            raw_score_df = pd.read_csv(match, sep='\t')
            rna_df = raw_score_df.dropna(subset=['RNA_score']).copy()
            rna_df['hgvs_c']=rna_df['hgvs_c'].transform(lambda x: x.split(':')[1])
            rna_dfs[gene]=rna_df

    return rna_dfs

In [ ]:
gene_paths= find_genes(input_directory)
rna_dfs = get_rna(rna_dir, gene_paths)

# Helper functions to build visualizations

In [ ]:
#Read all data into dictionary of dataframes
def read_data(gene):
    data_dict = {}
    paths = gene_paths[gene]

    to_read = ['del_counts', 'snv_counts', 'edit_rates']

    for elem in to_read:
        df = pd.read_csv(paths[elem], sep = '\t')

        if elem=='edit_rates':

            def recode_reps(df):

                if len(df) != 3:
                    df = df.assign(rep=['R1', 'R2'])

                return df
            
            df['target'] = df['target_rep'].transform(lambda x: x.split('_')[1].split('X')[1])
            df['rep'] = df['target_rep'].transform(lambda x: x.split('_')[2])

            rep_map = {'R1R4': 'R1',
                       'R1R2R3': 'R1',
                       'R2R5': 'R2',
                       'R4R5R6': 'R2',
                       'R3R6': 'R3',
                       'R7R8R9': 'R3'
                       }
            
            df['rep'] = df['rep'].map(rep_map)

            df = df.groupby('target').apply(recode_reps).reset_index(drop = True)
        
        if elem=='snv_counts':
            day_columns = [col for col in df.columns.tolist() if "D" in col]
            new_day_columns = [col.replace('_', ' ') for col in day_columns]
            
            new_col_names = dict(zip(day_columns, new_day_columns))

            df = df.rename(columns = new_col_names)
            
            df['max_count'] = df[new_day_columns].max(axis = 1)
            df = df.loc[df['max_count'] < 100000].copy()

        data_dict[elem] = df
    
    data_dict['scores'] = paths['scores_excel']
    
    return data_dict

In [ ]:
#Editing rate bar plot
def edit_rate_barplot(df):
    sort_order = natsorted(df['target'].tolist())
    plot = alt.Chart(df).mark_bar().encode(
        x = alt.X('rep',
                  axis = alt.Axis(
                      title = '',
                      labels = False,
                      ticks = False
                  )
                 ),
        y = alt.Y('edit_rate',
                  title = 'Editing Rate',
                  axis = alt.Axis(labelFontSize = 14,
                                  titleFontSize = 16
                  ),
                  scale = alt.Scale(domain = [0, 0.5])
                 ),
        column = alt.Column('target',
                            sort = sort_order,
                            header=alt.Header(title='SGE Target')
                           ),
        color = alt.Color('rep', 
                          legend = alt.Legend(
                              title = '',
                              titleFontSize = 16,
                              labelFontSize = 14,
                              labelFont = 'Arial',
                              titleFont = 'Arial',
                              orient = 'bottom'
                          )
                         )
    ).properties(
        width = 25,
        height = 200
    ).configure_facet(
        spacing = 5
    ).configure_axis(
        grid = False,
        labelFont = 'Arial',
        titleFont = 'Arial'
    ).configure_header(
        title = None,
        labelFontSize = 14,
        labelFont = 'Arial',
        titleFont = 'Arial'
    )

    plot.display()

    return plot

In [ ]:
# Correlation heatmap

def corr_heatmap(df):
    target_dfs = df.groupby('target')

    final_tuples = []
    for target, df in target_dfs:
        all_day_cols = [col for col in df.columns.tolist() if "D" in col]

        d05_cols = [col for col in all_day_cols if "D05" in col]
        d13_cols = [col for col in all_day_cols if "D13" in col]

        d17_cols = []
        if "D17 R1" in all_day_cols:
            d17_cols = [col for col in all_day_cols if "D17" in col]
            day_cols = [d05_cols, d13_cols, d17_cols]
        else:
            day_cols = [d05_cols, d13_cols]

        test_pairs_dict = {}
        for i, col_list in enumerate(day_cols):
            test_pairs = list(itertools.combinations(col_list, 2))

            day_map = {0: 'D5',
                       1: 'D13',
                       2: 'D17'}
            
            test_pairs_dict[day_map[i]] = test_pairs

        for day in test_pairs_dict.keys():
            tests_to_do = test_pairs_dict[day]

            for  rep1, rep2 in tests_to_do:
                if len(df.dropna(subset = [rep1, rep2])) == 0:
                    continue
                
                corr,_=stats.pearsonr(df[rep1], df[rep2])

                mod_target = target.split('_')[1].split('X')[1]
                data_tuple = (mod_target, f"{rep1} vs. {rep2.split(' ')[1]}", corr)
                final_tuples.append(data_tuple)
    
    corr_df = pd.DataFrame(final_tuples, columns = ['Targets', 'Tests', 'corr'])
    
    targets = set(corr_df['Targets'].tolist())
    targets = natsorted(targets)

    num_targets = len(targets)
    num_tests = len(corr_df['Tests'].unique())

    base = alt.Chart(corr_df, title = alt.TitleParams(text = '', fontSize = 32)).encode(
        x = alt.X('Tests:N'),
        y = alt.Y('Targets:N', sort = targets)
    )
    
    graph = base.mark_rect().encode(
                x = alt.X('Tests:N', axis = alt.Axis(title = '', titleFontSize = 28, labelFontSize = 24, labelLimit = 300, labelAngle = 45)),
                y = alt.Y('Targets', axis = alt.Axis(title = '', titleFontSize = 28, labelFontSize = 24), sort = targets),
                color = alt.Color('corr:Q', scale = alt.Scale(domain = [.2, 1]), legend = alt.Legend(title = "Pearson's r", titleFontSize = 24,labelFontSize = 22, labelFont = 'Arial', titleFont = 'Arial')),
                tooltip = [alt.Tooltip('corr', title = "Pearson's r: ")]
    ).properties(
        width = 70 * num_tests,
        height = 25 * num_targets
    )

    color = (
        alt.when(alt.datum.corr > 0.5)
        .then(alt.value("white"))
        .otherwise(alt.value("black"))
    )

    text = base.mark_text(baseline = 'middle', fontSize = 20).encode(
        text = alt.Text('corr:Q',format = "0.3f"), color = color
    ).transform_filter(
    'isValid(datum.corr)'
    )

    graph = (graph + text).configure_axis(
        grid = False,
        labelFont = 'Arial',
        titleFont = 'Arial'
    ).configure_view(
        stroke = None
    )
    
    graph.display()

    return graph




In [ ]:
# ClinVar precision-recall

def make_pr_curve(
    df: pd.DataFrame,
    score_col: str = "auth_reported_score",
    label_col: str = "simple_clinvar_sig_2025",
    pos_label: str = "Pathogenic",
    neg_label: str = "Benign",
    title: str = "Precision-Recall",
) -> alt.Chart:
    """Build a Precision-Recall curve chart using Altair.

    Rows with labels other than pos_label/neg_label (e.g. 'VUS')
    are excluded. Lower auth_reported_score = more likely abnormal (positive),
    so scores are negated before computing the curve.

    Parameters
    ----------
    df : DataFrame containing score_col and label_col.
    score_col : Column with continuous predictor scores.
    label_col : Column with functional class labels.
    pos_label : Label string treated as the positive class.
    neg_label : Label string treated as the negative class.
    title : Chart title prefix; AUC-PR is appended automatically.

    Returns
    -------
    alt.Chart
    """

    clinvar_vars = len(df.dropna(subset=['clinvar_sig_2025']))

    sub = df[df[label_col].isin([pos_label, neg_label])].copy()

    if len(sub) < 10:
        return None

    y_true = (sub[label_col] == pos_label).astype(int)
    # Negate: lower score = more abnormal, sklearn expects higher = more positive
    y_score = -sub[score_col]

    precision, recall, _ = precision_recall_curve(y_true, y_score)
    pr_auc = auc(recall, precision)
    baseline = y_true.mean()

    curve_df = pd.DataFrame({"Recall": recall, "Precision": precision})

    curve = (
        alt.Chart(curve_df, title=f"{title} (n = {clinvar_vars})")
        .mark_line(color="orange")
        .encode(
            x=alt.X("Recall:Q", scale=alt.Scale(domain=[0, 1]),
                    axis=alt.Axis(title="Recall", labelFontSize = 18, titleFontSize = 20, labelFont="Arial", titleFont="Arial")),
            y=alt.Y("Precision:Q", scale=alt.Scale(domain=[baseline * 0.95, 1]),
                    axis=alt.Axis(title="Precision", labelFontSize = 18, titleFontSize = 20, labelFont="Arial", titleFont="Arial")),
            tooltip=[
                alt.Tooltip("Recall:Q", format=".3f"),
                alt.Tooltip("Precision:Q", format=".3f"),
            ],
        )
        .properties(width=300, height=300)
    )

    baseline_df = pd.DataFrame({"y": [baseline]})
    baseline_rule = (
        alt.Chart(baseline_df)
        .mark_rule(color="gray", strokeDash=[4, 4])
        .encode(y="y:Q")
    )

    auc_text = alt.Chart(pd.DataFrame({
        'x': [0.05],
        'y': [baseline * 1.1],
        'text': [f'AUC = {pr_auc:.3f}']
    })).mark_text(
        align='left',
        baseline='bottom',
        fontSize=18,
        fontWeight='bold',
        color='black'
    ).encode(
        x=alt.X('x:Q', scale=alt.Scale(domain=[0, 1])),
        y=alt.Y('y:Q', scale=alt.Scale(domain=[baseline * 0.95, 1])),
        text='text:N'
    )

    final_plot = (curve + baseline_rule + auc_text).configure_title(font="Arial", fontSize=14).configure_axis(grid = False).configure_view(stroke = None)
    final_plot.display()
    return final_plot


In [ ]:
#ClinVar Strip Plot
def clinvar_strip(df, score_col='auth_reported_score', clinvar_col='clinvar_sig_2025', consequence_col='simplified_consequence'):

    df = df.dropna(subset=[clinvar_col]).copy()

    if len(df) < 10:
        return None
    #Color palette
    palette = [
    '#006616', # dark green,
    '#81B4C7', # dusty blue
    '#ffcd3a', # yellow
    '#6AA84F', # med green
    '#93C47D', # light green
    '#888888', # med gray
    '#000000', # black
    '#1170AA', # darker blue
    '#CFCFCF' # light gray  
    ]
    
    
    variant_types = [
        'Synonymous',
        'Missense',  
        'Stop Gained',
        'Intron', 
        'UTR Variant',
        'Stop Lost',
        'Start Lost',
        'Canonical Splice', 
        'Splice Region'
    ]
    
    order_of_interest = [
        'Synonymous',
        'Intron',
        'UTR Variant',
        'Stop Lost',
        'Missense',
        'Splice Region',
        'Start Lost',
        'Canonical Splice',
        'Stop Gained'
    ]
    
    df[consequence_col] = pd.Categorical(df[consequence_col], categories = order_of_interest, ordered = True)
    plp_strip = alt.Chart(df).mark_tick(opacity = 1, thickness = 2).encode(
                x = alt.X(score_col,
                            title = 'Fitness Score',
                            axis = alt.Axis(labelFontSize = 16, 
                                            titleFontSize = 20
                                            )
                            ),
                y = alt.Y(clinvar_col,
                            title = '',
                            sort = ['Benign', 'Likely Benign', 'VUS', 'Likely Pathogenic', 'Pathogenic'],
                            axis = alt.Axis(
                                labelFontSize = 16,
                                titleFontSize = 20,
                                labelLimit = 1000
                            )
                            ),
                order = alt.Order(consequence_col, sort = 'ascending'),
                color = alt.Color(consequence_col,
                                    scale = alt.Scale(
                                        range = palette,
                                        domain = variant_types
                                    ),
                                    legend = alt.Legend(titleFontSize = 16,
                                                        labelFontSize = 14,
                                                        title = 'Consequence'
                                                        )
                                    )
            ).properties(
                width = 600,
                height = 250,
                title = alt.TitleParams(text = f'ClinVar Variants in SGE (n = {len(df)})', fontSize = 22)
            )
    
    #Builds summary df with variant counts for each germline classification category
    summary_df = df[clinvar_col].value_counts().reset_index()
    summary_df['count'] = summary_df['count'].astype(str)
    summary_df['text'] = '(n = ' + summary_df['count'] + ')'


    #Lines for functional and non-functional cutoffs
    nf_line = alt.Chart(pd.DataFrame({'x': [df.loc[df['auth_reported_func_class']=='functionally_abnormal'][score_col].max()]})).mark_rule(color = 'black', strokeDash = [8,8], strokeWidth = 2).encode(
        x = 'x')

    func_lin = alt.Chart(pd.DataFrame({'x': [df.loc[df['auth_reported_func_class']=='functionally_normal'][score_col].min()]})).mark_rule(color = '#888888', strokeDash = [8,8], strokeWidth = 2).encode(
        x = 'x')
    
    #Builds y-axis labels with variant counts
    counts = alt.Chart(summary_df).mark_text(
        align='right',
        dx=-10,  # Slight offset to the left of the y-axis
        dy=20,  # Offset below the category label
        fontSize=16,
        color='black'
    ).encode(
        y=alt.Y(f'{clinvar_col}:N', sort = ['Benign', 'Likely benign', 'Uncertain significance', 'Likely pathogenic', 'Pathogenic']),
        text='text',
        x=alt.value(0)  # Position at the left edge
    )

    #Builds final strip plot
    plp_strip = (plp_strip + counts + nf_line + func_lin).configure_axis(
        grid = False
    ).configure_view(
        stroke = None
    )
    
    plp_strip.display()
    return plp_strip

In [ ]:
#RNA scatter plots
def rna_scatter(df, rna_df):

   final_df = pd.merge(df,rna_df,on='hgvs_c')
   final_df = spliceai_mapper(final_df)
   final_df = final_df[final_df['simplified_consequence'].isin(['Missense'])]

   #Color palette
   palette = [
   '#81B4C7', # dusty blue
   "#A2A2A2"
   ]
    
    
   variant_types = [
      'Missense',  
      'Splice Region'
   ]
   
   order = ['Donor Gain', 'Acceptor Gain', 'Donor Loss', 'Acceptor Loss', '< Threshold']

   final_df['splice_impact'] = pd.Categorical(final_df['splice_impact'], categories=order, ordered=True)

   scatter = alt.Chart(final_df).mark_circle().encode(
      x=alt.X('auth_reported_score:Q', title='Fitness Score', axis=alt.Axis(labelFontSize=14,titleFontSize=16)),
      y=alt.Y('RNA_score:Q', title='RNA Score', axis=alt.Axis(labelFontSize=14,titleFontSize=16)),
      order=alt.Order('splice_impact:N', sort='descending'),
      color = alt.Color('splice_impact:N',
                        legend=alt.Legend(
                           title='Predictied Splicing Impact',
                           orient='bottom',
                           columns=2
                        )
         )
      )

   #Lines for functional and non-functional cutoffs
   nf_line = alt.Chart(pd.DataFrame({'auth_reported_score': [df.loc[df['auth_reported_func_class']=='functionally_abnormal']['auth_reported_score'].max()]})).mark_rule(color = 'black', strokeDash = [8,8], strokeWidth = 2).encode(
      x = 'auth_reported_score')

   func_lin = alt.Chart(pd.DataFrame({'auth_reported_score': [df.loc[df['auth_reported_func_class']=='functionally_normal']['auth_reported_score'].min()]})).mark_rule(color = '#888888', strokeDash = [8,8], strokeWidth = 2).encode(
      x = 'auth_reported_score')
   
   scatter = (scatter + nf_line + func_lin).configure_axis(grid = False).configure_view(stroke = None )
   scatter.display()

   return scatter, final_df


In [ ]:
# Gene library cartoon
import sys
sys.path.insert(0, '/Users/ivan/Documents/GitHub/sge-utils/SimpleSGEViz')
from sgeviz import io as sge_io

CARTOON_DATA_DIR = Path('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/sge_data_for_qc/gene_cartoon_data')
DOMAINS_PATH     = CARTOON_DATA_DIR / '20260414_SGE_protein_domains.xlsx'
TARGETS_DIR      = CARTOON_DATA_DIR / 'targets_tsvs'

_DOMAIN_COLORS = [
    '#B9DBF4', '#C8DBC8', '#F6BF93', '#D5D0F2',
    '#018571', '#D35400', '#2980B9', '#C0392B',
]


def _build_vcoords(
    exon_df: pd.DataFrame,
    meta_df: pd.DataFrame,
    intron_vw: float = 0.04,
) -> tuple:
    """Map genomic exon coordinates to a normalised visual [0, ~1] x-axis.

    CDS exon widths are proportional to their genomic bp. UTR sub-segments
    and introns each receive *intron_vw* of visual space.

    Returns
    -------
    segments : list[dict]
        Each entry has keys: kind ('cds'|'utr'|'intron'), gstart, gend,
        vstart, vend, exon (str | None).
    gv : callable
        gv(genomic_pos) -> visual x-coordinate.
    """
    meta   = dict(zip(meta_df['type'], meta_df['info']))
    strand = str(meta.get('strand', 'plus')).lower()
    atg    = int(meta['atg'])
    stop   = int(meta['stop'])

    orig = exon_df.sort_values('start').reset_index(drop=True)
    if strand == 'minus':
        exons = orig.copy()
        exons['start'] = -orig['end'].values
        exons['end']   = -orig['start'].values
        atg, stop = -atg, -stop
    else:
        exons = orig.copy()
    exons = exons.sort_values('start').reset_index(drop=True)
    n = len(exons)

    def _split(gs: int, ge: int) -> list:
        subs, p = [], gs
        if p < atg:
            subs.append((p, min(ge, atg), 'utr')); p = min(ge, atg)
        if p < ge and p < stop:
            subs.append((p, min(ge, stop), 'cds')); p = min(ge, stop)
        if p < ge:
            subs.append((p, ge, 'utr'))
        return subs

    all_subs = [
        (i, g0, g1, k, str(row.get('exon', i + 1)))
        for i, row in exons.iterrows()
        for g0, g1, k in _split(int(row['start']), int(row['end']))
    ]

    utr_max_vw    = 3 * intron_vw
    utr_bps       = [g1 - g0 for _, g0, g1, k, __ in all_subs if k == 'utr']
    utr_px_per_bp = utr_max_vw / max(utr_bps) if utr_bps else 0.0
    utr_vw        = lambda bp: max(utr_px_per_bp * bp, intron_vw)

    cds_bp     = sum(g1 - g0 for _, g0, g1, k, __ in all_subs if k == 'cds')
    compressed = (n - 1) * intron_vw + sum(utr_vw(g1 - g0) for _, g0, g1, k, __ in all_subs if k == 'utr')
    cds_vw     = max(0.1, 1.0 - compressed)
    px_per_bp  = cds_vw / max(cds_bp, 1)

    segments: list = []
    vpos = 0.0
    prev_idx = None

    for exon_idx, g0, g1, kind, label in all_subs:
        if prev_idx is not None and exon_idx != prev_idx:
            i0 = int(exons.iloc[prev_idx]['end'])
            i1 = int(exons.iloc[exon_idx]['start'])
            segments.append({'kind': 'intron', 'gstart': i0, 'gend': i1,
                              'vstart': vpos, 'vend': vpos + intron_vw, 'exon': None})
            vpos += intron_vw

        vw = utr_vw(g1 - g0) if kind == 'utr' else px_per_bp * (g1 - g0)
        segments.append({'kind': kind, 'gstart': g0, 'gend': g1,
                         'vstart': vpos, 'vend': vpos + vw, 'exon': label})
        vpos += vw
        prev_idx = exon_idx

    def gv(gpos: float) -> float:
        """Interpolate a genomic position to its visual x-coordinate."""
        p = -gpos if strand == 'minus' else gpos
        for seg in segments:
            if seg['gstart'] <= p <= seg['gend']:
                span = seg['gend'] - seg['gstart']
                t = (p - seg['gstart']) / span if span else 0.0
                return seg['vstart'] + t * (seg['vend'] - seg['vstart'])
        return segments[0]['vstart'] if p < segments[0]['gstart'] else segments[-1]['vend']

    return segments, gv


def _build_aa_to_genomic_map(exon_df: pd.DataFrame, meta_df: pd.DataFrame):
    """Return a callable aa_to_gen(aa_pos) -> genomic coordinate.

    Walks CDS exons in transcription order, accumulating nucleotides, to map
    a 1-based amino acid position back to its first nucleotide's genomic
    coordinate. Handles both plus- and minus-strand genes.
    """
    meta   = dict(zip(meta_df['type'], meta_df['info']))
    strand = str(meta.get('strand', 'plus')).lower()
    atg    = int(meta['atg'])
    stop   = int(meta['stop'])
    exons  = exon_df.sort_values('start').reset_index(drop=True)

    if strand == 'plus':
        cds_segs = [
            (max(int(r['start']), atg), min(int(r['end']), stop))
            for _, r in exons.iterrows()
            if min(int(r['end']), stop) > max(int(r['start']), atg)
        ]
        def aa_to_gen(aa_pos):
            target, acc = (aa_pos - 1) * 3, 0
            for gs, ge in cds_segs:
                if acc + (ge - gs) > target:
                    return gs + (target - acc)
                acc += ge - gs
            return cds_segs[-1][1]

    else:  # minus: atg at high genomic coord, stop at low
        cds_segs = [
            (max(int(r['start']), stop), min(int(r['end']), atg))
            for _, r in exons.sort_values('start', ascending=False).iterrows()
            if min(int(r['end']), atg) > max(int(r['start']), stop)
        ]
        def aa_to_gen(aa_pos):
            target, acc = (aa_pos - 1) * 3, 0
            for cds_lo, cds_hi in cds_segs:
                if acc + (cds_hi - cds_lo) > target:
                    return cds_hi - (target - acc)
                acc += cds_hi - cds_lo
            return cds_segs[-1][0]

    return aa_to_gen


def load_targets_tsv(gene: str) -> pd.DataFrame:
    """Load library amplicons from the targets TSV, returning start/end columns."""
    t = pd.read_csv(TARGETS_DIR / f'{gene}.targets.tsv', sep='\t')
    return t[['editstart', 'editstop']].rename(columns={'editstart': 'start', 'editstop': 'end'})


def make_sge_gene_cartoon(
    gene: str,
    exon_df: pd.DataFrame,
    meta_df: pd.DataFrame,
    lib_df: pd.DataFrame,
    domain_df: pd.DataFrame | None = None,
    scores_df: pd.DataFrame | None = None,
    exon_color: str = '#d0d0d0',
    hatch: str = '//////',
    fig_width: float = 10,
) -> plt.Figure:
    """Draw a single-track SGE library cartoon.

    CDS exon blocks are colored by protein domain where applicable (neutral
    exon_color elsewhere). UTR sub-regions are drawn as thinner blocks.
    Introns are shown as backbone gaps. Library-amplicon coverage is
    crosshatched over CDS regions. ATG and Stop markers are drawn above.
    """
    segments, gv = _build_vcoords(exon_df, meta_df)
    total_vw  = segments[-1]['vend']
    meta      = dict(zip(meta_df['type'], meta_df['info']))
    has_dom   = domain_df is not None and not domain_df.empty

    # Y-coordinate constants
    BB          =  0.0          # backbone
    CB,  CT     = -0.28,  0.28  # CDS rect bottom / top
    UB,  UT     = -0.17,  0.17  # UTR rect bottom / top
    LBL         = -0.40         # exon number labels
    DOM_LBL     = -0.65         # domain name labels
    MKR         =  0.40         # ATG/Stop text

    y_bot = DOM_LBL - 0.08 if has_dom else LBL - 0.08
    fig, ax = plt.subplots(figsize=(fig_width, 1.5 if has_dom else 1.2))

    # Backbone: draw only across intron gaps so it never underlaps exon/UTR rects
    for seg in segments:
        if seg['kind'] == 'intron':
            ax.plot([seg['vstart'], seg['vend']], [BB, BB], color='black', lw=1.2, zorder=1)

    # Exon/UTR base blocks
    exon_vspans: dict = {}
    for seg in segments:
        if seg['kind'] == 'intron':
            continue
        name = seg['exon']
        exon_vspans.setdefault(name, [seg['vstart'], seg['vend']])
        exon_vspans[name][0] = min(exon_vspans[name][0], seg['vstart'])
        exon_vspans[name][1] = max(exon_vspans[name][1], seg['vend'])

        yb, yt = (CB, CT) if seg['kind'] == 'cds' else (UB, UT)
        ax.add_patch(mpatches.Rectangle(
            (seg['vstart'], yb), seg['vend'] - seg['vstart'], yt - yb,
            facecolor=exon_color, edgecolor='black', lw=0.8, zorder=2,
        ))

    # Domain coloring overlay on CDS segments
    if has_dom:
        aa_to_gen = _build_aa_to_genomic_map(exon_df, meta_df)
        for i, (_, dom) in enumerate(domain_df.iterrows()):
            color = _DOMAIN_COLORS[i % len(_DOMAIN_COLORS)]
            v0 = gv(aa_to_gen(int(dom['start'])))
            v1 = gv(aa_to_gen(int(dom['end'])))
            vlo, vhi = min(v0, v1), max(v0, v1)

            dom_v_extents = []
            for seg in segments:
                if seg['kind'] != 'cds':
                    continue
                ov0 = max(seg['vstart'], vlo)
                ov1 = min(seg['vend'],   vhi)
                if ov1 > ov0:
                    ax.add_patch(mpatches.Rectangle(
                        (ov0, CB), ov1 - ov0, CT - CB,
                        facecolor=color, edgecolor='black', lw=0.8, zorder=3,
                    ))
                    dom_v_extents.append((ov0, ov1))

            if dom_v_extents:
                label_x = (min(s[0] for s in dom_v_extents) + max(s[1] for s in dom_v_extents)) / 2
                ax.text(label_x, DOM_LBL, dom['domain'],
                        ha='center', va='top', fontsize=14, fontweight='bold', font='Arial')

    # Library coverage crosshatch (CDS and UTR, skipping introns)
    for _, row in lib_df.iterrows():
        v0 = gv(int(row['start']))
        v1 = gv(int(row['end']))
        vlo, vhi = min(v0, v1), max(v0, v1)
        for seg in segments:
            if seg['kind'] == 'intron':
                continue
            ov0 = max(seg['vstart'], vlo)
            ov1 = min(seg['vend'],   vhi)
            if ov1 > ov0:
                yb, yt = (CB, CT) if seg['kind'] == 'cds' else (UB, UT)
                ax.add_patch(mpatches.Rectangle(
                    (ov0, yb), ov1 - ov0, yt - yb,
                    facecolor='none', edgecolor='black', lw=0, hatch=hatch, zorder=4,
                ))

    # Exon labels
    for name, (vs, ve) in sorted(exon_vspans.items(), key=lambda kv: kv[1][0]):
        lbl = name[1:] if name[:1] == 'X' and name[1:].isdigit() else str(name)
        ax.text((vs + ve) / 2, LBL, lbl,
                ha='center', va='top', fontsize=14, fontweight='bold', font='Arial')

    # ATG / Stop markers
    for key, label in [('atg', 'ATG'), ('stop', 'Stop')]:
        if key in meta:
            vx = gv(int(meta[key]))
            ax.annotate('', xy=(vx, CT), xytext=(vx, MKR),
                        arrowprops=dict(arrowstyle='->', color='black', lw=1.0))
            ax.text(vx, MKR + 0.04, label, ha='center', va='bottom',
                    fontsize=12, fontweight='bold', color='black', font='Arial')

    # Gene label + optional stats
    ax.text(1.02, 0.75, gene, transform=ax.transAxes, clip_on=False,
            ha='left', va='center', fontsize=16, fontweight='bold', font='Arial')

    if scores_df is not None and not scores_df.empty:
        n_nts = scores_df['hg38_start'].nunique()
        n_snv = (scores_df['hg38_start'] == scores_df['hg38_end']).sum()
        n_del = ((scores_df['hg38_end'] - scores_df['hg38_start']).abs() == 3).sum()
        stats_text = f'{n_nts:,} nts targeted\n{n_snv:,} SNVs\n{n_del:,} 3-bp deletions'
        ax.text(1.02, 0.58, stats_text, transform=ax.transAxes, clip_on=False,
                ha='left', va='top', fontsize=14, font='Arial', linespacing=1.6)

    ax.set_xlim(-0.01, total_vw + 0.01)
    ax.set_ylim(y_bot, MKR + 0.25)
    ax.axis('off')

    plt.tight_layout()
    plt.show()
    fig.savefig(f'/Users/ivan/Desktop/pillar_project_figs/revision_qc_plots/sge/{gene}_LibCartoon.svg', format = 'svg')
    return fig

# Builds standard QC viz. for each gene

In [ ]:
annotated_rna_dfs = {}
for gene in gene_paths:

    print(f'\n------ {gene} ------')
    data_dict = read_data(gene)
    exon_df, _, meta_df = sge_io.fetch_exon_coords(gene)
    lib_df    = load_targets_tsv(gene)
    domain_df = pd.read_excel(DOMAINS_PATH, sheet_name=gene)

    make_sge_gene_cartoon(gene, exon_df, meta_df, lib_df, domain_df, scores_df=data_dict['scores'])

    
    edit_rate_plot = edit_rate_barplot(data_dict['edit_rates'])
    pearson_heatmap = corr_heatmap(data_dict['snv_counts'])
    plp_strip = clinvar_strip(data_dict['scores'])
    pr_curve = make_pr_curve(data_dict['scores'])

    rna_scatter_plot = None
    genes_w_rna=rna_dfs.keys()
    if gene in genes_w_rna:
        rna_df = rna_dfs[gene]
        rna_scatter_plot, annotated_rna_df = rna_scatter(data_dict['scores'], rna_df)
        annotated_rna_dfs[gene] = annotated_rna_df
    
    if rna_scatter_plot is not None:
        rna_scatter_plot.save(f'/Users/ivan/Desktop/pillar_project_figs/revision_qc_plots/sge/{gene}_rna_scatter.svg')

    edit_rate_plot.save(f'/Users/ivan/Desktop/pillar_project_figs/revision_qc_plots/sge/{gene}_editing_rate.svg')
    pearson_heatmap.save(f'/Users/ivan/Desktop/pillar_project_figs/revision_qc_plots/sge/{gene}_correlation.svg')

    if pr_curve and plp_strip is not None:
        plp_strip.save(f'/Users/ivan/Desktop/pillar_project_figs/revision_qc_plots/sge/{gene}_clinvar_strip.svg')
        pr_curve.save(f'/Users/ivan/Desktop/pillar_project_figs/revision_qc_plots/sge/{gene}_prcurve.svg')

# SGE vs. SpliceAI

In [ ]:
splice_df = all_scores.dropna(subset = ['maxSpliceAI'])
splice_df = splice_df.loc[splice_df['simplified_consequence'] == 'missense_variant']
splice_df = splice_df[['Gene', 'auth_reported_score', 'maxSpliceAI', 'simplified_consequence']]

splice_df.head()

In [ ]:
spliceai_scatter = alt.Chart(splice_df).mark_circle().encode(
    x = alt.X('maxSpliceAI:Q', title = 'Max SpliceAI Delta Score'),
    y = alt.Y('auth_reported_score:Q', title = 'SGE Fitness Score'),
    color = alt.Color('simplified_consequence:N',
                      scale = alt.Scale(
                          domain=['missense_variant'],
                          range=['#81B4C7']
                      ))
).properties(
    title = 'SGE vs. SpliceAI'
)

spliceai_scatter.display()

# Comparison to orthogonal experimental data

## Scatter plot helper function

In [ ]:
def scatter(df, col1, col2, splice_ai_threshold=0.2, no_splice=False):

    def scatterplot(df):

        r,_ = stats.pearsonr(df[col1], df[col2])

        no_splice_df = df.loc[df['maxSpliceAI'] < 0.2]

        no_splice_r,_ = stats.pearsonr(no_splice_df[col1], no_splice_df[col2])

        x_min = df[col1].min()
        y_min = df[col2].min()

        x_max = df[col1].max()
        y_max = df[col2].max()

        plot = alt.Chart(df).mark_circle().encode(
            x = alt.X(f"{col1}:Q",
                      scale = alt.Scale(domain = [1.05*x_min, 1.05*x_max])
                      ),
            y = alt.Y(f"{col2}:Q",
                      scale = alt.Scale(domain=[1.05*y_min, 1.05*y_max])),
            color = alt.Color("consequence:N",
                                legend=alt.Legend(
                                    orient = 'bottom',
                                    direction = 'horizontal',
                                    titleOrient = 'top',
                                    titleAnchor = 'start'
                                ),
                                scale = alt.Scale(
                                    domain = ['Missense'],
                                    range = ['#81B4C7']
                                )
                            ),
            stroke=alt.condition(
                alt.datum.maxSpliceAI > splice_ai_threshold,
                alt.value('black'),
                alt.value(None),
            ),
            strokeWidth=alt.condition(
                alt.datum.maxSpliceAI > splice_ai_threshold,
                alt.value(1.5),
                alt.value(0)
            ),
            order=alt.condition(
                alt.datum.maxSpliceAI > splice_ai_threshold,
                alt.value(1),
                alt.value(0)
            )
        )

        x_min = df[col1].min()
        y_max = df[col2].max()

        pearson_text = alt.Chart(pd.DataFrame({
                f'{col1}': [x_min, x_min],
                f'{col2}': [y_max * 0.95, y_max * 0.75],
                'text': [f'r = {r:.3f} (w/ SpliceAI > 0.2)', f'r = {no_splice_r:.3f} (w/o SpliceAI > 0.2)']
            })).mark_text(
                align='left',
                baseline='top',
                fontSize=12,
                fontWeight='bold',
                color='black'
            ).encode(
                x=alt.X(f'{col1}:Q', scale=alt.Scale(domain=[1.05*x_min, 1.05*x_max])),
                y=alt.Y(f'{col2}:Q', scale=alt.Scale(domain=[1.05*y_min, 1.05*y_max])),
                text='text:N'
            )
        
        plot = alt.layer(plot, pearson_text)

        return plot
    
    if no_splice:
        df = df.loc[df['maxSpliceAI'] < 0.2]
        plot = scatterplot(df)
    else:
        plot = scatterplot(df)


    

    return plot

## RAD51D Orthogonal Data

In [ ]:
rad51d_df = read_data('RAD51D')['scores'][['Gene', 'hg38_start', 'ref_allele', 'alt_allele', 'hgvs_c', 'hgvs_p', 'simplified_consequence', 'auth_reported_score', 'auth_reported_func_class', 'maxSpliceAI']]
rad51d_df = rad51d_df.rename(columns = {'hg38_start': 'pos',
                                        'ref_allele': 'ref',
                                        'alt_allele': 'alt',
                                        'simplified_consequence': 'consequence',
                                        'auth_reported_score': 'fitness_score',
                                        'auth_reported_func_class': 'fit_score_func_class'})
rad51d_df.head()

In [ ]:
darrah_mave_scores = pd.read_excel('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/sge_data_for_qc/orthogonal_sge_data/RAD51D_Darrah2026.xlsx', sheet_name="mave_scores")

darrah_mave_scores = darrah_mave_scores[['genomic_pos', 'ref', 'alt', 'HGVSc', 'HGVSp', 'aascore']]
darrah_mave_scores = darrah_mave_scores.rename(columns = {'genomic_pos': 'pos',
                                                          'HGVSc': 'hgvs_c',
                                                          'HGVSp': 'hgvs_p',
                                                          'aascore': 'mave_score'}
                                                            )

darrah_mave_scores = darrah_mave_scores.dropna(subset='mave_score')
darrah_mave_scores.head()

In [ ]:
rad51d_orthogonal = pd.read_excel('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/sge_data_for_qc/orthogonal_sge_data/RAD51D_Darrah2026.xlsx', sheet_name="orthogonal_assay_data")

rad51d_orthogonal.head()

In [ ]:
sge_mave_score = pd.merge(rad51d_df, darrah_mave_scores, on=['pos', 'hgvs_c', 'ref', 'alt', 'hgvs_p'], how='inner')

sge_mave_score = sge_mave_score.loc[sge_mave_score['consequence'] == 'Missense']
sge_mave_score.head()

In [ ]:
rad51d_mave_scatter = scatter(sge_mave_score, "fitness_score", "mave_score")

rad51d_mave_scatter.save('/Users/ivan/Desktop/pillar_project_figs/revision_qc_plots/sge/RAD51D_SGEvsDarrah.svg')
rad51d_mave_scatter.display()

In [ ]:
'''
ortho_assays=['perc_hr','perc_y2h_xrcc2']

ortho_assay_plots=[]

for assay in ortho_assays:
    plot=scatter(sge_orthogonal,assay, 'fitness_score')
    
    ortho_assay_plots.append(plot)

final_plot = alt.hconcat(*ortho_assay_plots).display()
'''

## PALB2 Orthogonal Data

In [ ]:
palb2_df = all_scores.loc[all_scores['Gene'] == 'PALB2'].copy()
palb2_df = palb2_df[['Gene', 'hg38_start', 'ref_allele', 'alt_allele', 'hgvs_c', 'hgvs_p', 'simplified_consequence', 'auth_reported_score', 'auth_reported_func_class', 'maxSpliceAI']]
palb2_df = palb2_df.rename(columns = {'hg38_start': 'pos',
                                        'ref_allele': 'ref',
                                        'alt_allele': 'alt',
                                        'simplified_consequence': 'consequence',
                                        'auth_reported_score': 'fitness_score',
                                        'auth_reported_func_class': 'fit_score_func_class'})


palb2_df.head()

In [ ]:
orthogonal_palb2 = pd.read_excel('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/sge_data_for_qc/orthogonal_sge_data/PALB2_Boonen2025.xlsx', sheet_name='scores')

orthogonal_palb2=orthogonal_palb2.loc[orthogonal_palb2['consequence']=='Missense'].copy()
orthogonal_palb2 = orthogonal_palb2.drop('consequence', axis=1)
orthogonal_palb2['hgvs_p']='p.' + orthogonal_palb2['aa_change']

orthogonal_palb2.head()

In [ ]:
palb2_merged=pd.merge(palb2_df,orthogonal_palb2, on='hgvs_p', how='inner')

palb2_merged.loc[palb2_merged['consequence']=='missense_variant', 'consequence'] = 'Missense'
palb2_merged.head()

In [ ]:
palb2_scatter = scatter(palb2_merged, 'fitness_score', 'score')

palb2_scatter.display()

In [ ]:
orthogonal_assays = ['dr-gfp', 'parpi_resistance']

palb2_orthogonal_plots = []
for assay in orthogonal_assays:
    df = palb2_merged.dropna(subset = assay)

    plot = scatter(df,assay, "fitness_score")
    
    palb2_orthogonal_plots.append(plot)

final_palb2_plot = alt.hconcat(*palb2_orthogonal_plots).display()

# High-level RNA Analysis

## Process external-RNA data

In [ ]:
vhl_data=pd.read_excel('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/sge_data_for_qc/raw_scores_for_rna/external_rna_data/2025_VHLBuckley.xlsx')
vhl_data = vhl_data.replace({'LOF1': 'LoF', 'LOF2':"LoF",
                             'STOP_GAINED': 'Nonsense',
                             'NON_SYNONYMOUS': 'Missense',
                             'SYNONYMOUS': 'Synonymous',
                             "CANONICAL_SPLICE":'Canonical splice',
                             'INTRONIC': 'Intronic',
                             "SPLICE_SITE": 'Splice region',
                             'STOP_LOST': 'Stop lost'})

vhl_data = vhl_data.dropna(subset=['function_class']).copy()

findlay_data=pd.read_excel('/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/sge_data_for_qc/raw_scores_for_rna/external_rna_data/20260422_BRCA1_Findlay2018_wRNA.xlsx')
findlay_data = molecular_consequence_mapper(findlay_data,'simplified_consequence')

findlay_data=findlay_data.rename(columns={'auth_reported_func_class': 'function_class',
                                          'simplified_consequence': 'Consequence',
                                          'mean.rna.score': 'rna_score'}
                                          )

findlay_consequence_map = {'FUNC':'Neutral',
                           'INT': 'Intermediate',
                           'LOF': 'LoF'
                           }

findlay_data['function_class'] = findlay_data['function_class'].map(findlay_consequence_map)
findlay_data.drop('clinvar_date_last_reviewed_2018', axis = 1)
findlay_data['maxSpliceAI']=findlay_data[['spliceAI_DS_AG', 'spliceAI_DS_AL', 'spliceAI_DS_DG', 'spliceAI_DS_DL']].max(axis = 1)

findlay_data = spliceai_mapper(findlay_data)

external_rna_data = {'VHL': vhl_data,
                     'BRCA1': findlay_data
                     }

In [ ]:
external_rna_data['VHL'].head()

In [ ]:
external_rna_data['BRCA1'].head()

In [ ]:
all_rna_df = pd.DataFrame()

rna_threshold_dict={
    'BARD1': -1.244,
    'RAD51D': -2.86,
    "VHL": -3,
    'BRCA1': -2

}
for gene in external_rna_data.keys():
    df = external_rna_data[gene]

    df['rna_stdev_consequence'] = 'normal'
    df.loc[df['rna_score'] <= rna_threshold_dict[gene], 'rna_stdev_consequence'] = 'low'

    summary = df.loc[(df['function_class']=='LoF') & (df['Consequence']=='Missense')]['rna_stdev_consequence'].value_counts(normalize=True).reset_index().iloc[[1]]

    summary['gene'] = gene

    summary=summary[['gene','rna_stdev_consequence', 'proportion']]
    all_rna_df = pd.concat([all_rna_df,summary])

print(all_rna_df)

## Process Internal RNA data

In [ ]:
for gene in rna_dfs.keys():
    df = rna_dfs[gene]

    df.loc[df['RNA_score'] <= rna_threshold_dict[gene], 'rna_stdev_consequence'] = 'low'

    summary = df.loc[(df['functional_consequence']=='functionally_abnormal') & (df['consequence']=='missense_variant')]['rna_stdev_consequence'].value_counts(normalize=True).reset_index().iloc[[1]]
    summary['gene']=gene
    summary=summary[['gene','rna_stdev_consequence', 'proportion']]
    all_rna_df = pd.concat([all_rna_df,summary])
print(all_rna_df)

## Visualize % Missense with Low RNA

In [ ]:
bars = alt.Chart(all_rna_df).mark_bar().encode(
    x=alt.X('gene:N', axis=alt.Axis(title='Gene')),
    y=alt.Y("proportion:Q", axis=alt.Axis(title='Proportion of LoF Missense Vars. w/ Low RNA')),
    color=alt.Color("gene:N", legend=None)
).properties(
    height=250,
    width=200
)

bars.save('/Users/ivan/Desktop/pillar_project_figs/revision_qc_plots/pathomechanism/LowRNAProportionLoFMissense.png', dpi=600)
bars.display()

# Wrangle Internal RNA data for Quantile analysis

In [ ]:
#Helper function to get quantiles
def get_quantiles(df, rna_score_col, missense_label='Missense', consequence_col='simplified_consequence', func_consequence_col='functional_consequence',lof_label='functionally_abnormal'):
    df = df.dropna(subset = [rna_score_col]).copy()

    df['percentile_class'] = pd.qcut(
        df[rna_score_col],
        q=4,
        labels=['<= 25th', '25th - 50th', '50th - 75th', '>= 75th'],
    )
    
    df = df.loc[(df[consequence_col]==missense_label) & (df[func_consequence_col]==lof_label)].copy()
    
    

    return df

In [ ]:
to_concat=[]
for gene in annotated_rna_dfs.keys():
    df = annotated_rna_dfs[gene]
    df=spliceai_mapper(df)
    df = get_quantiles(df, 'RNA_score')

    to_concat.append(df)

findlay_quantiles = get_quantiles(external_rna_data['BRCA1'], 'rna_score', consequence_col='Consequence',func_consequence_col='function_class', lof_label='LoF')
findlay_quantiles = findlay_quantiles[['Gene', 'maxSpliceAI', 'percentile_class', 'splice_impact']]

to_concat.append(findlay_quantiles)

final_df = pd.concat(to_concat)
final_df.head()

In [ ]:
vhl_df = external_rna_data['VHL']
vhl_df =vhl_df.rename(columns = {'max_spliceAI': 'maxSpliceAI',
                         'SpliceAI.acc.gain': 'spliceAI_DS_AG',
                         'SpliceAI.acc.loss': 'spliceAI_DS_AL',
                         'SpliceAI.don.gain': 'spliceAI_DS_DG',
                         'SpliceAI.don.loss': 'spliceAI_DS_DL'}
                         )

vhl_df = spliceai_mapper(vhl_df)

vhl_df = get_quantiles(vhl_df, 'rna_score', consequence_col='Consequence', func_consequence_col='function_class', lof_label='LoF')
vhl_df['Gene'] = 'VHL'



vhl_df = vhl_df[['Gene', 'maxSpliceAI', 'percentile_class', 'splice_impact']]

final_df = pd.concat([final_df, vhl_df])

In [ ]:
final_df.head()

In [ ]:
plot = alt.Chart(final_df).mark_circle().encode(
    x=alt.X('percentile_class:N', axis = alt.Axis(labelAngle=0), sort = ['<= 25th', '25th - 50th', '50th - 75th', '>= 75th']),
    xOffset=alt.XOffset('jitter:Q', scale=alt.Scale(domain=[-0.5, 0.5])),
    y=alt.Y('maxSpliceAI:Q'),
    color='splice_impact:N',
    tooltip=['Gene']
).transform_calculate(
    jitter='random() - 0.5'
).properties(
    height = 300,
    width = 500,
    title=f"Distribution of SpliceAI Score vs. RNA Score Percentile (n = {len(final_df)})"
)

plot.save('/Users/ivan/Desktop/pillar_project_figs/revision_qc_plots/pathomechanism/SpliceAIbyRNAQuartile.png',dpi=600)
plot.display()


In [ ]:
_, p = stats.mannwhitneyu(final_df.loc[final_df['percentile_class']=='<= 25th']['maxSpliceAI'], final_df.loc[final_df['percentile_class']=='>= 75th']['maxSpliceAI'])
print(p)

# Intron variant analysis

In [ ]:
sge_subset='/Users/ivan/Documents/GitHub/PillarProject_CAVA_Analysis/Data/20260101_SGEsubset.xlsx'
all_sge_df=pd.read_excel(sge_subset)

all_sge_df.head()

In [ ]:
all_sge_df = spliceai_mapper(all_sge_df)
all_sge_df = molecular_consequence_mapper(all_sge_df, 'simplified_consequence')

introns_only=all_sge_df.loc[all_sge_df['simplified_consequence']=='Intron']

In [ ]:
# Scatter plot helper function

def corr_scatter(df, rep1, rep2, gene):

    rep1_max = df[rep1].max(axis = 0)
    rep2_max = df[rep2].max(axis = 0)
    rep2_min = df[rep2].min(axis = 0)

    x_max = rep1_max * 1.05
    y_max = rep2_max * 1.05
    y_min = rep2_min * 1.05

    df = df.dropna(subset = [rep1, rep2]).copy()
    
    scatter = alt.Chart(df).mark_circle().encode(
        x = alt.X(f'{rep1}:Q',
                  scale = alt.Scale(0, x_max)
                  ),
        y = alt.Y(f'{rep2}:Q',
                  scale = alt.Scale(y_min, y_max)
                  ),
        color='splice_impact:N'
    )

    corr,_=stats.pearsonr(df[rep1], df[rep2])

    r_text = alt.Chart(pd.DataFrame({
        rep1: [x_max * 0.95],
        rep2: [1],
        'text': [f'r = {corr:.3f}']
    })).mark_text(
        align='right',
        baseline='bottom',
        fontSize=18,
        fontWeight='bold',
        color='black'
    ).encode(
        x = alt.X(f'{rep1}:Q',
                  scale = alt.Scale(0, x_max)),
        y = alt.Y(f'{rep2}:Q',
                  scale = alt.Scale(y_min, y_max)
                  ),
        text='text:N'
    )

    scatter = (scatter + r_text).properties(title = gene).resolve_scale(x = 'shared', y = 'shared')

    rep_test = f'{rep1} vs. {rep2}'
    return scatter, gene, rep_test, corr

In [ ]:
scatter, _, _, _ = corr_scatter(introns_only, 'auth_reported_score', 'maxSpliceAI', 'All SGE Intron Variants')
scatter.save('/Users/ivan/Desktop/pillar_project_figs/revision_qc_plots/pathomechanism/SpliceAI_vs_AllIntronVars.png',dpi=600)
scatter.display()

In [ ]:
scatter, _, _, _ = corr_scatter(introns_only.loc[introns_only['auth_reported_func_class']=='functionally_abnormal'], 'auth_reported_score', 'maxSpliceAI', 'LoF SGE Intron Variants')
scatter.save('/Users/ivan/Desktop/pillar_project_figs/revision_qc_plots/pathomechanism/SpliceAI_vs_LoFIntronVars.png',dpi=600)
scatter.display()